In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent
from langchain.tools import tool 

In [5]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap= 200)
data_chunk = splitter.split_documents(docs)

In [7]:
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store = InMemoryVectorStore.from_documents(
    documents=data_chunk,
    embedding=embedding
)

## Build Agent : Tool + LLM + Prompt

In [ ]:
@tool
def retriever_tool(query:str):
    # isse llm ko pata chalega ki iss tool kka kya kaam hai
    """    
    This tool can help to retrieve data from pdf having details of medical report of any person
    """

    print("tool Called: ",query)   ## to check agent has made call to this tool and the passed query

    docs = vector_store.similarity_search(query)

    context = ""

    for doc in docs:
        context += doc.page_content + "\n\n"


    return context


In [9]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [17]:
system_prompt = """ You are a assistant that answers the questions. Always use 'retriever_tool' tool for question requiring external knowledge"""


agent = create_agent(
    model= llm,
    tools = [retriever_tool],
    system_prompt=system_prompt
)

In [18]:
query = "what is the name of patient and who is doctor"
res = agent.invoke({"messages": [{"role":"user","content":query}]})

tool Called:  patient name and doctor name


In [16]:
print(res["messages"][-1].content[0]['text'])

Based on the medical report:

* **Patient Name:** Ms. Nikita Chudhary
* **Referring Doctor:** Dr. Nitin Nahar
